In [ ]:
"""
Script 5: Normality Tests
Perform Shapiro-Wilk, Anderson-Darling, and Kolmogorov-Smirnov tests
"""

import pandas as pd
import numpy as np
from scipy import stats as scipy_stats
from pathlib import Path

def shapiro_wilk_test(data, column_name):
    """Perform Shapiro-Wilk test for normality"""
    data_clean = data.dropna()
    
    # Shapiro-Wilk test
    statistic, p_value = scipy_stats.shapiro(data_clean)
    
    result = {
        'Test': 'Shapiro-Wilk',
        'Column': column_name,
        'Statistic': statistic,
        'P-Value': p_value,
        'Significant (α=0.05)': 'Yes' if p_value < 0.05 else 'No',
        'Interpretation': 'Not Normal' if p_value < 0.05 else 'Possibly Normal'
    }
    
    return result

def anderson_darling_test(data, column_name):
    """Perform Anderson-Darling test for normality"""
    data_clean = data.dropna()
    
    # Anderson-Darling test
    result_obj = scipy_stats.anderson(data_clean, dist='norm')
    
    # Extract results
    statistic = result_obj.statistic
    critical_values = result_obj.critical_values
    significance_levels = result_obj.significance_level
    
    # Check at 5% significance level
    critical_value_5pct = critical_values[2]  # 5% level
    is_normal = statistic < critical_value_5pct
    
    result = {
        'Test': 'Anderson-Darling',
        'Column': column_name,
        'Statistic': statistic,
        'Critical Value (5%)': critical_value_5pct,
        'Significant (α=0.05)': 'No' if is_normal else 'Yes',
        'Interpretation': 'Possibly Normal' if is_normal else 'Not Normal'
    }
    
    return result

def kolmogorov_smirnov_test(data, column_name):
    """Perform Kolmogorov-Smirnov test for normality"""
    data_clean = data.dropna()
    
    # Standardize data
    data_standardized = (data_clean - data_clean.mean()) / data_clean.std()
    
    # K-S test against standard normal distribution
    statistic, p_value = scipy_stats.kstest(data_standardized, 'norm')
    
    result = {
        'Test': 'Kolmogorov-Smirnov',
        'Column': column_name,
        'Statistic': statistic,
        'P-Value': p_value,
        'Significant (α=0.05)': 'Yes' if p_value < 0.05 else 'No',
        'Interpretation': 'Not Normal' if p_value < 0.05 else 'Possibly Normal'
    }
    
    return result

def jarque_bera_test(data, column_name):
    """Perform Jarque-Bera test for normality"""
    data_clean = data.dropna()
    
    # Jarque-Bera test
    statistic, p_value = scipy_stats.jarque_bera(data_clean)
    
    result = {
        'Test': 'Jarque-Bera',
        'Column': column_name,
        'Statistic': statistic,
        'P-Value': p_value,
        'Significant (α=0.05)': 'Yes' if p_value < 0.05 else 'No',
        'Interpretation': 'Not Normal' if p_value < 0.05 else 'Possibly Normal'
    }
    
    return result

def print_test_results(result):
    """Print test results in formatted way"""
    print(f"\n{result['Test']} Test - {result['Column']}")
    print("─" * 60)
    for key, value in result.items():
        if key not in ['Test', 'Column']:
            if isinstance(value, float):
                print(f"  {key:.<40} {value:.6f}")
            else:
                print(f"  {key:.<40} {value}")

def perform_normality_tests(df):
    """Perform all normality tests"""
    print("\n" + "="*70)
    print("NORMALITY TESTS")
    print("="*70)
    
    numeric_columns = ['Close Price', '3 Day MV.', '5 Day MV.']
    all_results = []
    
    for column in numeric_columns:
        if column in df.columns:
            data = df[column]
            
            # Shapiro-Wilk
            result_sw = shapiro_wilk_test(data, column)
            print_test_results(result_sw)
            all_results.append(result_sw)
            
            # Anderson-Darling
            result_ad = anderson_darling_test(data, column)
            print_test_results(result_ad)
            all_results.append(result_ad)
            
            # Kolmogorov-Smirnov
            result_ks = kolmogorov_smirnov_test(data, column)
            print_test_results(result_ks)
            all_results.append(result_ks)
            
            # Jarque-Bera
            result_jb = jarque_bera_test(data, column)
            print_test_results(result_jb)
            all_results.append(result_jb)
    
    # Create results dataframe
    results_df = pd.DataFrame(all_results)
    
    return results_df

def print_normality_summary(results_df):
    """Print summary of normality tests"""
    print("\n" + "="*70)
    print("NORMALITY TEST SUMMARY")
    print("="*70)
    
    for column in results_df['Column'].unique():
        column_results = results_df[results_df['Column'] == column]
        normal_count = (column_results['Interpretation'] == 'Possibly Normal').sum()
        total_tests = len(column_results)
        
        print(f"\n{column}:")
        print(f"  Tests indicating normality: {normal_count}/{total_tests}")
        
        if normal_count >= total_tests * 0.75:
            print(f"  Conclusion: Likely NORMAL distribution")
        elif normal_count >= total_tests * 0.5:
            print(f"  Conclusion: Mixed results - Possibly normal")
        else:
            print(f"  Conclusion: Likely NON-NORMAL distribution")

if __name__ == "__main__":
    file_path = 'outputs/cleaned_data.csv'
    
    try:
        df = pd.read_csv(file_path)
        df['Date'] = pd.to_datetime(df['Date'])
        
        # Perform normality tests
        results_df = perform_normality_tests(df)
        
        # Print summary
        print_normality_summary(results_df)
        
        # Save results
        Path('outputs').mkdir(exist_ok=True)
        results_df.to_csv('outputs/normality_tests.csv', index=False)
        print(f"\n✓ Normality test results saved to outputs/normality_tests.csv")
        print("✓ Normality tests completed!")
        
    except FileNotFoundError:
        print(f"Please run 01_data_loading.py first to generate {file_path}")